# v1 findings (4000 games, 769 input ANN, no quiescence)

**training** (val mse vs always predict train mean)

| round | labelled w | best val | baseline |
|---|---|---|---|
| 0 | material, depth 3 | 0.115 | 0.350 |
| 1 | nn r0, depth 2 | 0.037 | 0.252 |
| 2 | nn r1, depth 2 | 0.031 | 0.222 |
| 3 | nn r2, depth 2 | 0.030 | 0.210 |

- every round beats the baseline, way better than cool_heuristic (0.043 vs 0.047)
- BUT from round 1 the nn is predicting labels made by its own previous version, so low val mostly = agrees w itself, not = accurate

**sanity checks** (white pov, should be ~+0.9 / ~0 / ~-0.9)
- black down a queen: +0.75 (ok)
- start: +0.37 (should be ~0, leans white)
- white down a queen: -0.13 (bad, doesnt really count material)

**games vs material bot**
- notebook, depth 2, 20 games: +12 =5 -3 (72%), the 5 draws were just hitting the 200 ply cap
- terminal_game, depth 3, 5 games: +1 -4, at depth 3 material bot out calculates it

**issues**
1. tiny data: 15k positions for a 566k param net
2. errors pile up round to round (each net labels the next)
3. no flip: white + black patterns learnt separately, half the data effectively
4. noisy labels: search stops mid capture (horizon effect)
5. nn doesnt count material exactly, material bot does
6. matches were small and capped games counted as draws

**v2 next steps** (this notebook)
1. flip board to side to move + castling bits -> 772 inputs, labels from side to move pov
2. ~10x data: 40k games -> ~200k positions, nn runs in numpy in the workers
3. ANN2: y = tanh(nn + a * material / 4), nn only learns a correction on exact material
4. quiescence at the leaves (captures only) for cleaner labels
5. anti drift: rounds 1+ train on 0.75 * new labels + 0.25 * round 0 labels
6. bigger matches (40 games, depth 2 + 3, vs material + vs previous round), capped games decided by material
7. resumable: positions, labels, models saved to disk, finished rounds get skipped

# Goal in life (part 2)

- last time labels were game outcomes w discount, way too noisy, nn learnt basically nothing
- new idea: label positions w our OWN minmax score at depth d
- round 0: search w material function -> train nn
- round r: search w nn from round r-1 -> train nn again
- search + eval teaches the next eval, no stockfish involved (kinda TD-leaf / nnue vibes)

#Challenge 1:
- we still need positions, we steal them from lichess games but we IGNORE the results
- skip opening (everyone plays the same stuff), sample a few positions per game
- cached to csv so a restart doesnt redo this

In [1]:
# load up everything

import os
import chess
import numpy as np
import pandas as pd
import torch
import random
import re
from datasets import load_dataset
import time

num_games = 40000
skip_plies = 8
positions_per_game = 5
positions_path = "depthk2_positions.csv"

/Users/ahmed.khan/.pyenv/versions/3.12.3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def parse_moves(movetext):
    results = {"1-0", "0-1", "1/2-1/2", "*"}
    movetext = re.sub(r"\{[^}]*\}", " ", movetext)
    movetext = re.sub(r"\([^)]*\)", " ", movetext)
    tokens = movetext.split()
    final_tokens = []

    for t in tokens:
        if t.endswith("."):
            continue
        elif t.startswith("$"):
            continue
        elif t in results:
            continue
        else:
            final_tokens.append(t)
    return final_tokens

start_time = time.time()

if os.path.exists(positions_path): 
    positions = pd.read_csv(positions_path)

else:
    raw_chess_dataset = load_dataset("Lichess/standard-chess-games", split="train", streaming=True)

    random.seed(42)
    rows = []

    for game_id, game in enumerate(raw_chess_dataset.take(num_games)): #ranidmly take 5000 games

        board = chess.Board()
        fens = []
        for index, move in enumerate(parse_moves(game['movetext'])):
            board.push_san(move)
            if index >= skip_plies and not board.is_game_over(): ## skip first 8 moves cuz theyre too noisy (no pieces gain/lost so OUR eval learns nth)
                fens.append(board.fen())

        for fen in random.sample(fens, min(positions_per_game, len(fens))): #sample either 5 positons or all positions for small boards sample
            rows.append({"game_id": game_id, "fen": fen})

    positions = pd.DataFrame(rows) # we now have a diverse set of positions from many games (4000), with a few positions per game (3-5) to train on

print(len(positions), "positions from", positions['game_id'].nunique(), "games")
positions.head()
end_time = time.time()
print("Time taken to process positions:", end_time - start_time, "seconds")

195578 positions from 39295 games
Time taken to process positions: 0.08168315887451172 seconds


In [3]:

start_time = time.time()
game_ids = positions['game_id'].unique() # get all unique game IDs
np.random.seed(42) 
np.random.shuffle(game_ids)


cutoff = int(0.8 * len(game_ids))
train_games = set(game_ids[:cutoff]) # splitting by games NOT positions 


train_fens = positions[positions['game_id'].isin(train_games)]['fen'].tolist()
val_fens = positions[~positions['game_id'].isin(train_games)]['fen'].tolist()
print(len(train_fens), len(val_fens))
end_time = time.time()
print("Time taken to split train/val:", end_time - start_time, "seconds")

156421 39157
Time taken to split train/val: 0.020337820053100586 seconds


#Challenge 2:
- labelling is SLOW (python minmax), so we use all cores
- worker stuff lives in depthk_worker.py cuz windows multiprocessing cant pickle notebook functions
- encode flips the board so the nn always sees it from the side to move, label is side to move pov too
- round 0 label = tanh(material search score / 4), nn rounds = nn search score (already in [-1, 1]), mate = +-1
- X stored as int8 (one hot + castling bits are 0/1), 200k x 772 floats would be ~600mb

In [4]:
from multiprocessing import Pool
from depthk_worker import encode, label_fen, play_game, get_eval
from ANN import ANN2
import time

def label_fens(fens, d, max_captures, model_path):    # takes frvr
    with Pool() as pool:
        args = [(fen, d, max_captures, model_path) for fen in fens]
        return pool.map(label_fen, args, chunksize=64)

def fens_to_tensor(fens):
    return torch.stack([encode(chess.Board(fen), torch.int8) for fen in fens])

start_time = time.time()

X_train = fens_to_tensor(train_fens)
X_val = fens_to_tensor(val_fens)
print(X_train.shape, X_val.shape)

end_time = time.time()
print("Time taken to encode fens:", end_time - start_time, "seconds")

torch.Size([156421, 772]) torch.Size([39157, 772])
Time taken to encode fens: 10.466519832611084 seconds


In [5]:
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn

# same as cool_heuristic train, plus weight decay, int8 -> float cast, keeps best weights

def train(model, X_train, Y_train, X_val, Y_val, epochs, batch_size, lr, patience, weight_decay=1e-5):

    device = None

    if torch.cuda.is_available():
        device = torch.device("cuda")
        print("Using CUDA")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Using MPS")
    else:
        device = torch.device("cpu")
        print("Using CPU")

    model.to(device)

    global_train_error = []
    global_val_error = []
    train_dataset = TensorDataset(X_train, Y_train)
    val_dataset = TensorDataset(X_val, Y_val)

    mse = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False)


    best_val = float("inf")
    best_state = None
    epochs_without_improvement = 0

    for i in range(epochs):
        epoch_train_errors = []
        epoch_val_errors = []

        model.train()
        for x_true, y_true in train_loader:
            optimizer.zero_grad()
            x_true = x_true.to(device)
            y_true = y_true.to(device)      
            y_hat = model(x_true.float()).squeeze(-1)
            train_loss = mse(y_hat, y_true)
            epoch_train_errors.append(train_loss.item())
            train_loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            for x_true, y_true in val_loader:
                x_true = x_true.to(device)
                y_true = y_true.to(device)
                y_hat = model(x_true.float()).squeeze(-1)
                epoch_val_errors.append(mse(y_hat, y_true).item())

        train_mean = sum(epoch_train_errors) / len(epoch_train_errors)
        val_mean = sum(epoch_val_errors) / len(epoch_val_errors)
        global_train_error.append(train_mean)
        global_val_error.append(val_mean)

        # early stopping
        if val_mean < best_val:
            best_val = val_mean
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        print(f"epoch {i+1}/{epochs} | train {train_mean:.4f} | val {val_mean:.4f}")

        if epochs_without_improvement >= patience:
            print(f"early stop: no val improvement for {patience} epochs (best val {best_val:.4f})")
            break

    model.load_state_dict(best_state)
    return global_train_error, global_val_error

In [6]:
# sanity checks, printed from whites pov, should be ~+0.9 / ~0 / ~-0.9 regardless of who is to move

tests = {
    "black down a queen (w)": "rnb1kbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1",
    "black down a queen (b)": "rnb1kbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR b KQkq - 0 1",
    "start":                  chess.STARTING_FEN,
    "white down a queen (w)": "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNB1KBNR w KQkq - 0 1",
    "white down a queen (b)": "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNB1KBNR b KQkq - 0 1",
}

def sanity(model_path):
    eval_fn = get_eval(model_path)
    for name, fen in tests.items():
        print(f"  {name:24s} {eval_fn(chess.Board(fen), chess.WHITE):+.3f}")

#The loop
- search is base.Quiescence (captures + check evasions at the leaves)
- round 0: material + quiescence (usually upto 4 captures deep) at depth 2
- round 1+: nn + quiescence (2 captures deep, nn eval is slower) at depth 2
- rounds 1+ mix in round 0 labels so the nn doesnt drift off on its own errors (mix = 1.0 turns this off)
- warm start each round from the last model
- each experiment saves to its own folder (experiment_dir), experiment-1 = old MinMax search, mix 0.75
- ~40 min labelling per nn round on 8 cores (torch eval, slower than the old numpy one)
- every round is saved, rerunning skips finished rounds

in the first iteration we use a simple quessience search to get value of board state for depth `depth` + `material_captures`. we use this to create labels for 



In [7]:
import time

num_rounds = 4
depth = 2
material_captures = 4
nn_captures = 2
mix = 1.0
experiment_dir = "experiment-2"

os.makedirs(f"{experiment_dir}/model-weights", exist_ok=True)
os.makedirs(f"{experiment_dir}/intermediary-numpy-labels", exist_ok=True)

prev_path = None
history = []

for r in range(num_rounds):
    # we will needa modfiy model path and labels path for each round to avoid overwriting previous rounds and experiment with different depths and captures
    model_path  = f"{experiment_dir}/model-weights/chess_depthk2_r{r}.pt"
    labels_path = f"{experiment_dir}/intermediary-numpy-labels/depthk2_labels_r{r}.npz"


    if os.path.exists(labels_path):
        labels = np.load(labels_path)
        Y_train_new, Y_val_new = labels['train'], labels['val']

    else:
        t0 = time.time()
        captures = material_captures if r == 0 else nn_captures
        Y_train_new = np.array(label_fens(train_fens, depth, captures, prev_path), dtype=np.float32)
        Y_val_new   = np.array(label_fens(val_fens, depth, captures, prev_path), dtype=np.float32)
        np.savez(labels_path, train=Y_train_new, val=Y_val_new)
        print(f"round {r}: labelled in {time.time() - t0:.0f}s")

    if r == 0:
        Y0_train, Y0_val = Y_train_new, Y_val_new
        Y_train, Y_val = torch.tensor(Y_train_new), torch.tensor(Y_val_new)

    else:
        Y_train = torch.tensor(mix * Y_train_new + (1 - mix) * Y0_train)
        Y_val   = torch.tensor(mix * Y_val_new + (1 - mix) * Y0_val)

    baseline = ((Y_val - Y_train.mean()) ** 2).mean().item()   # always predict train mean

    model = ANN2()
    if prev_path is not None: model.load_state_dict(torch.load(prev_path))

    if os.path.exists(model_path):
        model.load_state_dict(torch.load(model_path))
        model.eval()
        with torch.no_grad():
            best_val = ((model(X_val.float()).squeeze(-1) - Y_val) ** 2).mean().item()
        print(f"round {r}: already trained, skipping")

    else:
        train_errors, val_errors = train(model, X_train, Y_train, X_val, Y_val, epochs=100, batch_size=512, lr=1e-4, patience=5)
        best_val = min(val_errors)
        model.cpu()
        torch.save(model.state_dict(), model_path)

    history.append({"round": r, "baseline": baseline, "best_val": best_val, "a": model.a.item()})
    print(f"round {r}: best val {best_val:.4f} vs baseline {baseline:.4f}, material weight a = {model.a.item():.3f}")
    sanity(model_path)
    print()
    prev_path = model_path

pd.DataFrame(history)

round 0: labelled in 822s
Using MPS
epoch 1/100 | train 0.0672 | val 0.0662
epoch 2/100 | train 0.0615 | val 0.0629
epoch 3/100 | train 0.0588 | val 0.0615
epoch 4/100 | train 0.0561 | val 0.0617
epoch 5/100 | train 0.0523 | val 0.0596
epoch 6/100 | train 0.0476 | val 0.0594
epoch 7/100 | train 0.0427 | val 0.0597
epoch 8/100 | train 0.0382 | val 0.0612
epoch 9/100 | train 0.0343 | val 0.0609
epoch 10/100 | train 0.0308 | val 0.0618
epoch 11/100 | train 0.0280 | val 0.0614
early stop: no val improvement for 5 epochs (best val 0.0594)
round 0: best val 0.0594 vs baseline 0.2811, material weight a = 0.906
  black down a queen (w)   +0.964
  black down a queen (b)   +0.963
  start                    +0.004
  white down a queen (w)   -0.963
  white down a queen (b)   -0.964

round 1: labelled in 2775s
Using MPS
epoch 1/100 | train 0.0347 | val 0.0489
epoch 2/100 | train 0.0315 | val 0.0483
epoch 3/100 | train 0.0289 | val 0.0488
epoch 4/100 | train 0.0262 | val 0.0491
epoch 5/100 | train 0

,round,baseline,best_val,a
0,0,0.281085,0.059377,0.905560
1,1,0.259598,0.048300,0.884745
2,2,0.256624,0.045443,0.874314
3,3,0.257011,0.043273,0.864578


#Does it actually play
- final nn vs material bot at depth 2 and 3, and vs the previous round at depth 2
- 40 games each, random 4-8 move openings, colours swapped, run in parallel
- both sides use base.Quiescence w nn_captures so its just eval vs eval
- capped at 200 plies, capped games go to whoever is up material

In [8]:
from base import Quiescence

def match(nn_spec, opp_spec, depth, n_games=40, max_plies=200):
    jobs = []
    for g in range(n_games):
        nn_white = g % 2 == 0
        white, black = (nn_spec, opp_spec) if nn_white else (opp_spec, nn_spec)
        jobs.append((white, black, depth, g, max_plies, Quiescence, nn_captures))

    with Pool() as pool:
        results = pool.map(play_game, jobs)

    wins = draws = losses = capped = 0
    for g, (result, was_capped) in enumerate(results):
        nn_white = g % 2 == 0
        capped += was_capped
        if result == "1/2-1/2": draws += 1
        elif (result == "1-0") == nn_white: wins += 1
        else: losses += 1
    score = (wins + 0.5 * draws) / n_games
    return {"wins": wins, "draws": draws, "losses": losses, "capped": capped, "score": score}

final_path = f"{experiment_dir}/model-weights/chess_depthk2_r{num_rounds - 1}.pt"
previous_path = f"{experiment_dir}/model-weights/chess_depthk2_r{num_rounds - 2}.pt"

matches = {
    "vs material, depth 2": match(final_path, None, 2),
    "vs material, depth 3": match(final_path, None, 3),
    "vs previous round, depth 2": match(final_path, previous_path, 2),
}
pd.DataFrame(matches).T

,wins,draws,losses,capped,score
"vs material, depth 2",9.0,1.0,30.0,23.0,0.2375
"vs material, depth 3",24.0,2.0,14.0,15.0,0.6250
"vs previous round, depth 2",20.0,0.0,20.0,11.0,0.5000
